# Frame VLM Semantic Factors Extractor v1 (Offline Kaggle Edition)

Notebook trích xuất thông tin ngữ nghĩa có cấu trúc (Structured Semantic Factors) từ video keyframes bằng Vision-Language Model **Qwen3-VL-30B-A3B-Instruct (BF16)** chạy hoàn toàn trong chế độ **Offline (Internet OFF)** trên Kaggle với GPU **NVIDIA RTX Pro 6000 Blackwell (96GB VRAM)**.

### Mô hình lựa chọn: `Qwen/Qwen3-VL-30B-A3B-Instruct`
- **Kiến trúc MoE (Mixture-of-Experts)**: Tổng 30 tỷ tham số, nhưng chỉ kích hoạt **~3.3 tỷ tham số** cho mỗi token sinh ra.
- **Tốc độ sinh text JSON siêu nhanh**: Nhanh gấp **3x – 4x** so với mô hình Dense 32B, tiết kiệm hàng chục giờ khi quét 311.000 keyframes.
- **Tận dụng tối đa băng thông GDDR7 của Blackwell**: Băng thông bộ nhớ > 1.5 TB/s của RTX Pro 6000 giải quyết triệt để bài toán định tuyến expert của MoE.

### Tích hợp trực tiếp Kaggle Dataset (Remote Source: `gs://aic_ai_2026/processed` mirror)
Notebook này đọc trực tiếp dữ liệu từ Kaggle Dataset đã gắn:
- `keyframes/dataset=ai_challenge_2025/batch=L21/` ... `batch=L30/` (hơn 311k files)
- `keyframes_manifests/dataset=ai_challenge_2025/batch=.../profile=autoshot_v1/shot_segments.csv`
- **Zero-download & Zero-network**: Toàn bộ frame ảnh và file manifest được đọc thẳng từ ổ đĩa NVMe của Kaggle `/kaggle/input/...`, loại bỏ 100% độ trễ mạng và không cần kết nối GCS.

### Schema 7 trường Semantic Factors
1. **`subjects`**: Thực thể chính, con người, vai trò, nhóm người (ví dụ: `["man in uniform", "group of students"]`).
2. **`actions`**: Hành động quan sát được, tương tác, cử chỉ (ví dụ: `["walking across street", "talking on phone"]`).
3. **`objects`**: Vật thể, xe cộ, đồ đạc, công cụ (ví dụ: `["motorcycle", "traffic light", "laptop"]`).
4. **`attributes`**: Màu sắc, số lượng, trang phục, hình dáng (ví dụ: `["red helmet", "blue jacket"]`).
5. **`scene`**: Bối cảnh, địa điểm, môi trường xung quanh (ví dụ: `["busy urban street", "coffee shop interior"]`).
6. **`text_cues`**: Text OCR nhìn thấy trên màn hình, biển hiệu, thương hiệu (nguyên bản, ví dụ: `["HIGHLANDS COFFEE", "DỪNG LẠI"]`).
7. **`temporal_context`**: Thời điểm, ánh sáng, thời tiết, môi trường (ví dụ: `["daytime", "bright sunlight", "indoor"]`).

### Dual-Contract & Tương thích hệ thống
- `kind`: `"semantic_factors"`
- `json_value`: Dictionary 7 trường đầy đủ
- `detected_objects`: Gộp `subjects` + `objects`
- `ocr_texts`: Gán từ `text_cues`
- `caption`: Tóm tắt tổng hợp các factors
- `text_value`: Ghép multiline tối ưu cho BM25 Full-text search và Embedding / Reranking
- `confidence`: `1.0` (hoặc `0.5` nếu fallback)


In [1]:
# =============================================================================
# 1. OFFLINE ENVIRONMENT & WHEEL INSTALLATION
# =============================================================================
from __future__ import annotations
import os, sys, subprocess, importlib.util
from pathlib import Path

# Enforce offline environment: no HuggingFace Hub or external PyPI network calls
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['TRANSFORMERS_LOCAL_FILES_ONLY'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

def discover_wheel_dir() -> Path | None:
    search_roots = [Path('/kaggle/input'), Path.cwd(), Path('/kaggle/working')]
    for root in search_roots:
        if not root.exists(): continue
        for p in root.rglob('*wheel*'):
            if p.is_dir() and any(p.glob('*.whl')):
                return p
    return None

# WHEEL_DIR = discover_wheel_dir()
WHEEL_DIR = Path("/kaggle/input/datasets/nguyentranthienan/wheels-qwen25-vl")
print('Discovered offline wheel directory:', WHEEL_DIR)

needs_install = []
if importlib.util.find_spec('qwen_vl_utils') is None:
    needs_install.append('qwen_vl_utils')
if importlib.util.find_spec('accelerate') is None:
    needs_install.append('accelerate')

# Check transformers version (Qwen3-VL and Qwen2.5-VL require transformers>=4.49.0)
try:
    import transformers
    from packaging import version
    if version.parse(transformers.__version__) < version.parse('4.49.0'):
        print(f'Installed transformers ({transformers.__version__}) is older than 4.49.0. Upgrading from wheels...')
        needs_install.append('transformers>=4.49.0')
except Exception:
    needs_install.append('transformers>=4.49.0')

if needs_install:
    print(f'Installing packages from offline wheels: {needs_install}')
    if WHEEL_DIR:
        subprocess.run([
            sys.executable, '-m', 'pip', 'install',
            '--no-index', '--no-cache-dir',
            '--find-links', str(WHEEL_DIR),
            *needs_install
        ], check=True)
    else:
        print('WARNING: WHEEL_DIR not found; expecting packages to be preinstalled.')

import transformers
import qwen_vl_utils
print(f'✓ transformers version: {transformers.__version__}')
print(f'✓ qwen_vl_utils available: {qwen_vl_utils.__file__}')


Discovered offline wheel directory: /kaggle/input/datasets/nguyentranthienan/wheels-qwen25-vl
Installing packages from offline wheels: ['qwen_vl_utils']
Looking in links: /kaggle/input/datasets/nguyentranthienan/wheels-qwen25-vl
Processing /kaggle/input/datasets/nguyentranthienan/wheels-qwen25-vl/qwen_vl_utils-0.0.14-py3-none-any.whl
Processing /kaggle/input/datasets/nguyentranthienan/wheels-qwen25-vl/av-18.1.0-cp311-abi3-manylinux_2_28_x86_64.whl (from qwen_vl_utils)
✓ transformers version: 5.0.0
✓ qwen_vl_utils available: /usr/local/lib/python3.12/dist-packages/qwen_vl_utils/__init__.py


In [2]:
# =============================================================================
# 2. PARAMETERS & MODEL SELECTION
# =============================================================================
from __future__ import annotations
from types import SimpleNamespace
from pathlib import Path
import torch

DATASET_ID = "ai_challenge_2025"
PROFILE_VERSION = "autoshot_v1"
# BATCHES = ["L21", "L22"]  # Run batches: ["L21"] or ["L21", "L22", ...]
# BATCHES = ["L23", "L24"]  # Run batches: ["L21"] or ["L21", "L22", ...]
# BATCHES = ["L25"]  # Run batches: ["L21"] or ["L21", "L22", ...]
BATCHES = ["L27", "L28"]  # Run batches: ["L21"] or ["L21", "L22", ...]
# BATCHES = ["L29", "L30"]  # Run batches: ["L21"] or ["L21", "L22", ...]

# Dataset paths: Auto-discover Kaggle dataset matching gs://aic_ai_2026/processed mirror
KAGGLE_DATASET_ROOT = "/kaggle/input/datasets/lcdngthnh/aic-2026"  # If empty, automatically discovers /kaggle/input/<dataset>
LOCAL_MODEL_PATH = "/kaggle/input/models/qwen-lm/qwen-3-vl/transformers/8b-instruct/1"     # If empty, automatically discovers model snapshot under /kaggle/input

# Extractor identity
OUTPUT_PREFIX = "features/extractors"
EXTRACTOR_NAME = "semantic_factors"
EXTRACTOR_VERSION = "fe-vlm-factors-v1-offline"
ANNOTATION_VERSION = "fe-vlm-factors-v1"

# Structured VLM Prompt (Max-Accuracy & Fine-Grained Object Detection Edition)
VLM_FACTORS_PROMPT = """You are an elite visual perception and open-vocabulary object detection engine specialized in fine-grained video indexing and retrieval.
Examine this video frame with high precision across foreground, background, and occluded areas.
Extract all observable semantic and visual factors strictly grounded in the image pixels.
Return ONLY a single valid JSON object containing EXACTLY these 7 keys:
1. "subjects": A comprehensive list of visible people, animals, and primary agents with their specific roles, identities, and distinguishing attire (e.g. ["traffic police officer in neon vest", "female delivery courier wearing green helmet", "elderly man with eyeglasses", "male pedestrian carrying black umbrella"]). Distinguish individuals clearly. If none, return [].
2. "actions": A detailed list of all observable bodily movements, postures, mechanical motions, and physical interactions between subjects and objects (e.g. ["riding electric scooter through puddle", "handing brown cardboard box to customer", "unlocking metal roll-up door", "operating smartphone with right hand", "sitting on red plastic stool"]). If completely static, state visible postures.
3. "objects": An exhaustive, fine-grained inventory of all prominent and contextual physical objects, vehicles, devices, tools, furniture, and infrastructure (e.g. ["white Honda Wave scooter", "traffic signal showing red light", "POS payment terminal", "security surveillance camera", "yellow safety barricade", "stainless steel cooking pot", "motorcycle rear license plate"]). Specify types, models, or materials where identifiable. Avoid generic labels like "thing" or "item".
4. "attributes": Prominent visual attributes, bound colors, patterns, physical counts, states, and condition descriptors (e.g. ["bright yellow reflective stripes", "metallic silver paint", "two motorcycles in motion", "broken glass pane", "wet asphalt surface", "striped cotton t-shirt", "cardboard material"]). Explicitly pair colors and counts with their corresponding targets.
5. "scene": Rich descriptors of the specific physical environment, spatial layout, architectural setting, and location type (e.g. ["narrow Vietnamese urban alleyway with hanging electric cables", "convenience store checkout counter", "multi-lane elevated highway interchange", "indoor wet market seafood stall", "sidewalk coffee shop with low plastic seating"]).
6. "text_cues": Verbatim transcription of ALL readable on-screen texts, shop banners, street signs, directional markers, vehicle license plates, brand logos, badges, timestamps, and broadcast channel bugs (e.g. ["PHỞ BÒ GIA TRUYỀN", "ĐƯỜNG NGUYỄN HUỆ", "59-X3 888.88", "HIGHLANDS COFFEE", "VTV1 HD", "CẤM ĐỖ XE"]). 
   - Retain EXACT original casing, digits, and Vietnamese diacritics.
   - Do NOT translate text into English.
   - If absolutely no readable text is present, return []. Never output ["none"] or descriptions.
7. "temporal_context": Precise ambient lighting, natural/artificial illumination sources, weather conditions, time-of-day indicators, and indoor/outdoor status (e.g. ["overcast daytime with diffused natural lighting", "nighttime illuminated by streetlamps and neon signs", "heavy tropical downpour with visible raindrops", "indoor warm fluorescent ceiling lights", "golden hour sunset glare"]).
Grounding & Extraction Rules:
- STRICT GROUNDING: Detect ONLY elements visually confirmed in the frame. Do NOT hallucinate, infer invisible internal components, or guess unseen entities.
- COMPOUND DESCRIPTORS: Prefer bound, descriptive terms ("red delivery scooter") over detached generic words ("car", "red").
- LANGUAGE: Keys 1, 2, 3, 4, 5, and 7 MUST be written in natural, descriptive English. Key 6 ("text_cues") MUST preserve exact original languages and scripts.
- FORMAT: Output pure, valid JSON ONLY. Do NOT enclose in markdown code fences (` ```json `), and do NOT add any conversational preamble or postscript.
""".strip()

# Model profiles
MODEL_REGISTRY = {
    "qwen3_vl_30b_a3b": {
        "backend": "qwen3_vl",
        "name": "Qwen3-VL-30B-A3B-Instruct",
        "hf_id": "Qwen/Qwen3-VL-30B-A3B-Instruct",
        "batch_size": 4,
        "max_new_tokens": 512,
        "torch_dtype": "bfloat16",
        "quantization": "none",
        "max_pixels": 1024 * 1024,
        "min_pixels": 256 * 256,
        "preprocess_workers": 4,
        "hints": ["30b-a3b", "qwen3-vl", "qwen-3-vl", "qwen-3", "qwen3", "qwen", "30b"],
    },
    "qwen3_vl_8b": {
        "backend": "qwen3_vl",
        "name": "Qwen3-VL-8B-Instruct",
        "hf_id": "Qwen/Qwen3-VL-8B-Instruct",
        "batch_size": 32,
        "max_new_tokens": 512,
        "torch_dtype": "bfloat16",
        "quantization": "none",
        "max_pixels": 1024 * 1024,
        "min_pixels": 256 * 256,
        "preprocess_workers": 4,
        "hints": ["8b", "qwen3-vl-8b", "qwen-3-vl", "qwen3", "qwen"],
    },
    "qwen3_vl_4b": {
        "backend": "qwen3_vl",
        "name": "Qwen3-VL-4B-Instruct",
        "hf_id": "Qwen/Qwen3-VL-4B-Instruct",
        "batch_size": 96,
        "max_new_tokens": 512,
        "torch_dtype": "bfloat16",
        "quantization": "none",
        "max_pixels": 1024 * 1024,
        "min_pixels": 256 * 256,
        "preprocess_workers": 4,
        "hints": ["4b", "qwen3-vl-4b", "qwen-3-vl", "qwen3", "qwen"],
    },
    "qwen25_vl_7b": {
        "backend": "qwen25_vl",
        "name": "Qwen2.5-VL-7B-Instruct",
        "hf_id": "Qwen/Qwen2.5-VL-7B-Instruct",
        "batch_size": 16,
        "max_new_tokens": 512,
        "torch_dtype": "bfloat16",
        "quantization": "none",
        "max_pixels": 1024 * 1024,
        "min_pixels": 256 * 256,
        "preprocess_workers": 4,
        "hints": ["7b", "qwen2.5-vl-7b", "qwen2.5-vl", "qwen"],
    },
    "qwen25_vl_32b": {
        "backend": "qwen25_vl",
        "name": "Qwen2.5-VL-32B-Instruct",
        "hf_id": "Qwen/Qwen2.5-VL-32B-Instruct",
        "batch_size": 4,
        "max_new_tokens": 512,
        "torch_dtype": "bfloat16",
        "quantization": "none",
        "max_pixels": 1024 * 1024,
        "min_pixels": 256 * 256,
        "preprocess_workers": 4,
        "hints": ["32b", "qwen2.5-vl", "qwen"],
    },
}

# Active model: Qwen3-VL-8B-Instruct (Dense BF16)
# ACTIVE_MODEL_KEY = "qwen3_vl_8b"
ACTIVE_MODEL_KEY = "qwen3_vl_4b"
BENCHMARK_MODEL_KEYS = [ACTIVE_MODEL_KEY]

DEVICE = "auto"
RUN_ROOT = "/kaggle/working/feature_extractor_runs"
UPLOAD_TO_GCS = False  # Pure offline execution

# Execution limits
DRY_RUN_MAX_FRAMES = 20
DEMO_BATCHES = ["L21"]
DEMO_MAX_FRAMES = 32
BENCHMARK_MAX_FRAMES = 16
FULL_MAX_FRAMES = None
CONFIRM_FULL_RUN = "RUN_FULL_DATASET"  # Set to "RUN_FULL_DATASET" before full run

# Local resume settings
SKIP_EXISTING = True
OVERWRITE = False
RESUME_ANNOTATIONS_PATH = ""  # Optional path to local annotations.jsonl

PIPELINE_BATCH_SIZE = None
LOG_EVERY_N_FRAMES = 64
USE_TQDM = True

cfg = SimpleNamespace(**{name: value for name, value in globals().copy().items() if name.isupper() and not name.startswith("_")})
print("Parameters loaded for Offline Frame VLM Semantic Factors Extractor.")
print("Active model profile:", cfg.ACTIVE_MODEL_KEY, "->", cfg.MODEL_REGISTRY[cfg.ACTIVE_MODEL_KEY]["name"])
print("Batches:", cfg.BATCHES, "Offline mode: ACTIVE (UPLOAD_TO_GCS=False)")

Parameters loaded for Offline Frame VLM Semantic Factors Extractor.
Active model profile: qwen3_vl_4b -> Qwen3-VL-4B-Instruct
Batches: ['L27', 'L28'] Offline mode: ACTIVE (UPLOAD_TO_GCS=False)


In [3]:
# =============================================================================
# 3. OFFLINE DATASET DISCOVERY & MANIFEST HELPERS
# =============================================================================
from __future__ import annotations
import csv, json, logging, os, re, shutil, time, uuid
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import Any
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm

@dataclass
class RunLayout:
    run_id: str
    run_dir: Path
    artifacts_dir: Path
    annotations_path: Path
    errors_path: Path
    metrics_path: Path
    summary_path: Path
    log_path: Path

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")

def cfg_value(config: Any, name: str, default: Any = None) -> Any:
    return getattr(config, name, default)

def make_run_id(kind: str) -> str:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    return f"{kind}_{stamp}_{uuid.uuid4().hex[:8]}"

def make_run_layout(config: Any, run_kind: str) -> RunLayout:
    run_id = make_run_id(run_kind)
    run_dir = Path(str(cfg_value(config, "RUN_ROOT", "/kaggle/working/feature_extractor_runs"))) / run_id
    artifacts_dir = run_dir / "artifacts"
    artifacts_dir.mkdir(parents=True, exist_ok=True)
    return RunLayout(
        run_id=run_id,
        run_dir=run_dir,
        artifacts_dir=artifacts_dir,
        annotations_path=artifacts_dir / "annotations.jsonl",
        errors_path=artifacts_dir / "errors.jsonl",
        metrics_path=artifacts_dir / "metrics.csv",
        summary_path=artifacts_dir / "summary.json",
        log_path=run_dir / "run.log",
    )

def setup_logging(layout: RunLayout) -> logging.Logger:
    logger = logging.getLogger("vlm_factors_offline")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    fmt = logging.Formatter("%(asctime)s %(levelname)s %(message)s")
    stream = logging.StreamHandler()
    stream.setFormatter(fmt)
    file_handler = logging.FileHandler(layout.log_path, encoding="utf-8")
    file_handler.setFormatter(fmt)
    logger.addHandler(stream)
    logger.addHandler(file_handler)
    return logger

def write_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

def append_jsonl(path: Path, records: list[dict]) -> int:
    with path.open("a", encoding="utf-8") as handle:
        for r in records:
            handle.write(json.dumps(r, ensure_ascii=False) + "\n")
    return len(records)

def append_metric(path: Path, row: dict) -> None:
    exists = path.exists()
    with path.open("a", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(row.keys()))
        if not exists:
            writer.writeheader()
        writer.writerow(row)

def iter_batches(items: list, batch_size: int):
    batch_size = max(1, int(batch_size))
    for start in range(0, len(items), batch_size):
        yield items[start:start + batch_size]

# ---------------------------------------------------------------------------
# Discovery for Kaggle Dataset (gs://aic_ai_2026/processed mirror)
# ---------------------------------------------------------------------------
def discover_kaggle_dataset_root(config: Any) -> Path:
    explicit = str(cfg_value(config, "KAGGLE_DATASET_ROOT", "") or "").strip()
    if explicit:
        p = Path(explicit).expanduser().resolve()
        if p.is_dir():
            return p
        raise FileNotFoundError(f"Configured KAGGLE_DATASET_ROOT not found: {explicit}")

    search_roots = [Path("/kaggle/input"), Path.cwd(), Path("/kaggle/working")]
    for root in search_roots:
        if not root.exists(): continue
        if (root / "keyframes").is_dir() or (root / "keyframes_manifests").is_dir():
            return root
        for sub in root.iterdir():
            if sub.is_dir() and ((sub / "keyframes").is_dir() or (sub / "keyframes_manifests").is_dir()):
                return sub
            if sub.is_dir():
                for nested in sub.iterdir():
                    if nested.is_dir() and ((nested / "keyframes").is_dir() or (nested / "keyframes_manifests").is_dir()):
                        return nested

    return Path("/kaggle/input")

def resolve_offline_image_path(dataset_root: Path, image_gcs_uri: str, record: dict) -> Path | None:
    """Resolve gs://aic_ai_2026/processed/keyframes/... to local disk path in Kaggle dataset."""
    # 1. Direct stripped mapping: 'processed/' -> dataset_root / ...
    if "processed/" in image_gcs_uri:
        rel = image_gcs_uri.split("processed/", 1)[1]
        cand = dataset_root / rel
        if cand.is_file():
            return cand

    # 2. Match standard keyframes layout: keyframes/dataset=.../batch=.../profile=.../video_id/frame.jpg
    cand = (
        dataset_root
        / "keyframes"
        / f"dataset={record['dataset_id']}"
        / f"batch={record['batch_id']}"
        / f"profile={record['profile_version']}"
        / record["video_id"]
        / Path(image_gcs_uri).name
    )
    if cand.is_file():
        return cand

    # 3. Direct match by image_rel_path
    if record.get("image_rel_path"):
        cand = dataset_root / "keyframes" / record["image_rel_path"]
        if cand.is_file():
            return cand

    # 4. Search under batch folder if present
    batch_dir = dataset_root / "keyframes" / f"dataset={record['dataset_id']}" / f"batch={record['batch_id']}"
    if batch_dir.is_dir():
        cand = batch_dir / record["video_id"] / Path(image_gcs_uri).name
        if cand.is_file():
            return cand
        for vid_dir in batch_dir.rglob(record["video_id"]):
            if vid_dir.is_dir():
                c = vid_dir / Path(image_gcs_uri).name
                if c.is_file():
                    return c

    return cand

def find_offline_manifest_path(dataset_root: Path, config: Any, batch_id: str) -> Path | None:
    """Find shot_segments.csv for one batch under keyframes_manifests/, prioritizing run_id=full_*."""
    dataset_id = str(cfg_value(config, "DATASET_ID", "ai_challenge_2025"))
    profile = str(cfg_value(config, "PROFILE_VERSION", "autoshot_v1"))
    b_id = str(batch_id).strip()
    b_id_lower = b_id.lower()

    base = dataset_root / "keyframes_manifests"
    if not base.is_dir():
        cands = [p for p in dataset_root.rglob("keyframes_manifests") if p.is_dir()]
        if cands:
            base = cands[0]
        else:
            return None

    # Collect candidate manifest files
    candidates = []
    
    # 1. Search under batch subdirectories
    search_dirs = [
        base / f"dataset={dataset_id}" / f"batch={b_id}" / f"profile={profile}",
        base / f"dataset={dataset_id}" / f"batch={b_id}",
        base / f"batch={b_id}",
    ]
    for bd in search_dirs:
        if bd.is_dir():
            for f in bd.rglob("shot_segments.csv"):
                if f.is_file(): candidates.append(f)
            for f in bd.rglob("*.csv"):
                if f.is_file(): candidates.append(f)
            for f in bd.rglob("*.jsonl"):
                if f.is_file(): candidates.append(f)

    # 2. General recursive search in base matching batch_id
    if not candidates:
        for f in base.rglob("*"):
            if not f.is_file() or f.suffix.lower() not in (".csv", ".jsonl"):
                continue
            p_str = f.as_posix().lower()
            name_str = f.name.lower()
            if (f"batch={b_id_lower}" in p_str or 
                f"/{b_id_lower}/" in p_str or 
                f"_{b_id_lower}." in p_str or 
                f"/{b_id_lower}." in p_str or
                name_str.startswith(f"{b_id_lower}_") or
                name_str.startswith(f"{b_id_lower}.")):
                candidates.append(f)

    if not candidates:
        return None

    candidates = list(set(candidates))

    # Prioritize:
    # 1. "full" run (e.g. run_id=full_autoshot_l21_...)
    # 2. Penalize demo_ and smoke_
    # 3. Prefer shot_segments.csv
    # 4. Prefer larger file size
    def score_candidate(cand: Path) -> tuple:
        p_str = cand.as_posix().lower()
        score = 0
        if "run_id=full" in p_str or "full_" in p_str or "/full/" in p_str:
            score += 1000
        if "demo" in p_str:
            score -= 300
        if "smoke" in p_str:
            score -= 500
        if cand.name == "shot_segments.csv":
            score += 100
        if f"profile={profile.lower()}" in p_str:
            score += 50
        file_size = cand.stat().st_size if cand.exists() else 0
        return (score, file_size)

    candidates.sort(key=score_candidate, reverse=True)
    return candidates[0]

def read_offline_manifest(path: Path) -> pd.DataFrame:
    if path.suffix == ".csv":
        return pd.read_csv(path)
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    return pd.DataFrame(rows)

def discover_offline_frame_records(config: Any, batches: list[str], max_frames: int | None = None) -> tuple[Path, list[dict]]:
    """Discover keyframe records from local Kaggle dataset mirror without network access."""
    dataset_root = discover_kaggle_dataset_root(config)
    dataset_id = str(cfg_value(config, "DATASET_ID", "ai_challenge_2025"))
    profile_version = str(cfg_value(config, "PROFILE_VERSION", "autoshot_v1"))
    all_records: list[dict] = []
    
    for batch_id in batches:
        manifest_path = find_offline_manifest_path(dataset_root, config, batch_id)
        if manifest_path is not None and manifest_path.is_file():
            df = read_offline_manifest(manifest_path)
            if "saved" in df.columns:
                df = df[df["saved"].astype(str).str.lower().isin(["true", "1", "yes"])]
            
            print(f"Loaded full manifest for batch={batch_id}: {manifest_path.parent.name}/{manifest_path.name} ({len(df)} frames)")
            
            for _, row in df.iterrows():
                image_gcs_uri = str(row.get("image_gcs_uri") or row.get("gcs_uri") or row.get("image_uri") or "").strip()
                if not image_gcs_uri:
                    continue
                video_id = str(row.get("video_id") or Path(image_gcs_uri).parent.name).strip()
                frame_idx = int(float(row.get("frame_idx", 0) or 0))
                keyframe_id = str(row.get("keyframe_id") or f"{video_id}_F{frame_idx:06d}")
                
                rec = {
                    "dataset_id": str(row.get("dataset_id") or dataset_id),
                    "batch_id": str(row.get("batch_id") or batch_id).strip(),
                    "video_id": video_id,
                    "video_name": str(row.get("video_name") or ""),
                    "shot_id": str(row.get("shot_id") or ""),
                    "frame_idx": frame_idx,
                    "frame_sec": float(row.get("frame_sec", row.get("timestamp", 0)) or 0),
                    "timestamp_ms": int(float(row.get("frame_sec", 0) or 0) * 1000),
                    "keyframe_id": keyframe_id,
                    "image_rel_path": str(row.get("image_rel_path") or f"{video_id}/{Path(image_gcs_uri).name}"),
                    "image_gcs_uri": image_gcs_uri,
                    "profile_version": str(row.get("profile_version") or profile_version),
                }
                local_path = resolve_offline_image_path(dataset_root, image_gcs_uri, rec)
                rec["local_image_path"] = str(local_path) if local_path else ""
                all_records.append(rec)
        else:
            # Manifest not found: synthesize records directly from image files under keyframes/
            search_dirs = [
                dataset_root / "keyframes" / f"dataset={dataset_id}" / f"batch={batch_id}",
                dataset_root / "keyframes" / f"batch={batch_id}",
            ]
            found_dir = next((d for d in search_dirs if d.is_dir()), None)
            if not found_dir:
                for cand in (dataset_root / "keyframes").rglob(f"*batch={batch_id}*"):
                    if cand.is_dir():
                        found_dir = cand
                        break
            if found_dir:
                img_files = []
                for ext in ("*.jpg", "*.jpeg", "*.png", "*.webp"):
                    img_files.extend(found_dir.rglob(ext))
                img_files.sort()
                
                print(f"Notice: Manifest not found for batch={batch_id}. Scanned {len(img_files)} images directly from {found_dir}...")
                
                for img_p in img_files:
                    vid = img_p.parent.name
                    if "=" in vid:
                        vid = img_p.stem.split("_")[0]
                        
                    num_match = re.search(r"(\d+)", img_p.stem)
                    frame_idx = int(num_match.group(1)) if num_match else 0
                    frame_sec = round(frame_idx / 25.0, 3)
                    
                    keyframe_id = img_p.stem if img_p.stem.startswith(vid) else f"{vid}_{img_p.stem}"
                    
                    rec = {
                        "dataset_id": dataset_id,
                        "batch_id": batch_id,
                        "video_id": vid,
                        "video_name": vid,
                        "shot_id": "",
                        "frame_idx": frame_idx,
                        "frame_sec": frame_sec,
                        "timestamp_ms": int(frame_sec * 1000),
                        "keyframe_id": keyframe_id,
                        "image_rel_path": f"{vid}/{img_p.name}",
                        "image_gcs_uri": f"gs://aic_ai_2026/processed/keyframes/dataset={dataset_id}/batch={batch_id}/profile={profile_version}/{vid}/{img_p.name}",
                        "profile_version": profile_version,
                        "local_image_path": str(img_p),
                    }
                    all_records.append(rec)
            
    all_records.sort(key=lambda r: (r["batch_id"], r["video_id"], r["frame_idx"], r["keyframe_id"]))
    if max_frames is not None:
        all_records = all_records[:int(max_frames)]
    return dataset_root, all_records

def load_processed_keyframes_local(config: Any) -> set[str]:
    """Load keyframe IDs already processed from previous local annotations.jsonl."""
    if not cfg_value(config, "SKIP_EXISTING", True) or cfg_value(config, "OVERWRITE", False):
        return set()
    
    explicit = str(cfg_value(config, "RESUME_ANNOTATIONS_PATH", "") or "").strip()
    candidates = []
    if explicit and Path(explicit).is_file():
        candidates.append(Path(explicit))
    
    run_root = Path(str(cfg_value(config, "RUN_ROOT", "/kaggle/working/feature_extractor_runs")))
    if run_root.is_dir():
        candidates.extend(sorted(run_root.rglob("annotations.jsonl"), key=lambda p: p.stat().st_mtime, reverse=True))
    
    if not candidates:
        return set()
    
    processed = set()
    for line in candidates[0].read_text(encoding="utf-8").splitlines():
        if not line.strip(): continue
        try:
            row = json.loads(line)
            if not row.get("error") and row.get("keyframe_id"):
                processed.add(str(row["keyframe_id"]))
        except Exception:
            pass
    return processed

def base_annotation(record: dict, config: Any, kind: str, run_id: str) -> dict:
    return {
        "dataset_id": record["dataset_id"],
        "batch_id": record["batch_id"],
        "video_id": record["video_id"],
        "keyframe_id": record["keyframe_id"],
        "frame_id": record["keyframe_id"],
        "shot_id": record.get("shot_id", ""),
        "frame_idx": record["frame_idx"],
        "frame_sec": record["frame_sec"],
        "timestamp_ms": record.get("timestamp_ms", int(float(record.get("frame_sec", 0)) * 1000)),
        "image_gcs_uri": record["image_gcs_uri"],
        "kind": kind,
        "caption": None,
        "ocr_texts": [],
        "detected_objects": [],
        "object_counts": {},
        "detections": [],
        "text_value": None,
        "json_value": {},
        "confidence": 1.0,
        "model_version": str(cfg_value(config, "MODEL_VERSION", "unknown")),
        "annotation_version": str(cfg_value(config, "ANNOTATION_VERSION", "fe-vlm-factors-v1")),
        "run_id": run_id,
        "created_at": utc_now(),
    }

def dry_run(config: Any, max_frames: int | None = None) -> dict:
    batches = [str(batch).upper() for batch in cfg_value(config, "BATCHES", [])]
    root, records = discover_offline_frame_records(config, batches, max_frames=max_frames)
    valid_paths = sum(1 for r in records if r.get("local_image_path") and Path(r["local_image_path"]).is_file())
    return {
        "status": "OFFLINE_DRY_RUN_OK",
        "dataset_root": str(root),
        "batches": batches,
        "planned_frames": len(records),
        "valid_local_image_paths": valid_paths,
        "sample_records": records[:3],
    }


In [4]:
# =============================================================================
# 4. TASK MODEL LOADING & EXTRACTION LOGIC (QWEN3-VL / QWEN2.5-VL BF16)
# =============================================================================
from __future__ import annotations
from typing import Any
import gc       
import torch
import logging

def init_logger(name: str = "vlm_extractor") -> logging.Logger:
    logger = logging.getLogger(name)
    if not logger.handlers:
        logger.setLevel(logging.INFO)
        h = logging.StreamHandler()
        h.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
        logger.addHandler(h)
    return logger

def _first_model_device(model: Any) -> Any:
    if hasattr(model, "device"):
        return model.device
    for param in model.parameters():
        return param.device
    return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

def _move_batch_to_device(inputs: Any, device: Any, dtype: Any | None = None):
    moved = {}
    for key, value in inputs.items():
        if torch.is_tensor(value):
            kwargs = {"device": device, "non_blocking": True}
            if dtype is not None and torch.is_floating_point(value):
                kwargs["dtype"] = dtype
            moved[key] = value.to(**kwargs)
        else:
            moved[key] = value
    return moved

def _image_uri(image_path: str | Path) -> str:
    # Do NOT use Path.as_uri() because as_uri() percent-encodes characters like "=" into "%3D",
    # which breaks qwen_vl_utils fetch_image (it opens image[7:] without URL unquoting).
    resolved = str(Path(image_path).expanduser().resolve())
    return f"file://{resolved}"

def _build_qwen_messages(image_path: str | Path, spec: dict) -> list[dict]:
    image_content: dict = {"type": "image", "image": _image_uri(image_path)}
    if spec.get("max_pixels"):
        image_content["max_pixels"] = int(spec["max_pixels"])
    if spec.get("min_pixels"):
        image_content["min_pixels"] = int(spec["min_pixels"])
    return [{"role": "user", "content": [image_content, {"type": "text", "text": str(spec["prompt"])}]}]

def _normalize_factors(data: dict) -> dict:
    keys = ["subjects", "actions", "objects", "attributes", "scene", "text_cues", "temporal_context"]
    normalized = {}
    for key in keys:
        val = data.get(key, [])
        if isinstance(val, list):
            normalized[key] = [str(x).strip() for x in val if x is not None and str(x).strip()]
        elif isinstance(val, str) and val.strip():
            normalized[key] = [val.strip()]
        else:
            normalized[key] = []
    return normalized

def safe_parse_json(raw_text: str) -> dict:
    empty_schema = {
        "subjects": [], "actions": [], "objects": [], "attributes": [],
        "scene": [], "text_cues": [], "temporal_context": []
    }
    if not raw_text or not raw_text.strip():
        res = dict(empty_schema)
        res["_parse_error"] = True
        return res

    text = raw_text.strip()
    if "<think>" in text:
        text = re.sub(r"<think>[\s\S]*?</think>", "", text).strip()
        text = re.sub(r"<think>[\s\S]*$", "", text).strip()

    if "```" in text:
        match = re.search(r"```(?:json)?\s*([\s\S]*?)\s*```", text, re.IGNORECASE)
        if match:
            text = match.group(1).strip()

    try:
        data = json.loads(text)
        if isinstance(data, dict):
            return _normalize_factors(data)
    except Exception:
        pass

    brace_match = re.search(r"\{[\s\S]*\}", text)
    if brace_match:
        try:
            data = json.loads(brace_match.group(0))
            if isinstance(data, dict):
                return _normalize_factors(data)
        except Exception:
            pass

    res = dict(empty_schema)
    res["_parse_error"] = True
    res["_raw_response"] = raw_text[:500]
    return res

def resolve_local_model_path(config: Any, model_key: str) -> Path:
    explicit = str(cfg_value(config, "LOCAL_MODEL_PATH", "") or "").strip()
    if explicit:
        p = Path(explicit).expanduser().resolve()
        if p.exists():
            return p
        raise FileNotFoundError(f"Configured LOCAL_MODEL_PATH not found: {explicit}")

    registry = cfg_value(config, "MODEL_REGISTRY", {})
    spec = registry.get(model_key, {})
    raw_hints = spec.get("hints", [model_key])
    norm_hints = [h.lower().replace("-", "").replace("_", "") for h in raw_hints]

    search_roots = [
        Path("/kaggle/input"),
        Path.cwd(),
        Path("/kaggle/working"),
        Path("/kaggle/input/models"),
    ]

    candidates = []
    for root in search_roots:
        if not root.exists():
            continue
        for p in root.rglob("*"):
            if not p.is_dir():
                continue
            has_weights = (p / "model.safetensors.index.json").is_file() or any(p.glob("*.safetensors"))
            has_cfg = (p / "config.json").is_file()
            if not (has_weights and has_cfg):
                continue

            p_norm = p.as_posix().lower().replace("-", "").replace("_", "")
            score = 0
            for h in norm_hints:
                if h in p_norm:
                    score += 10

            is_input = 1 if str(p).startswith("/kaggle/input") else 0
            candidates.append((score, is_input, -len(p.parts), p))

    if candidates:
        candidates.sort(reverse=True)
        chosen = candidates[0][3]
        return chosen

    for root in search_roots:
        if not root.exists():
            continue
        for p in root.rglob("config.json"):
            parent = p.parent
            has_w = any(parent.glob("*.safetensors")) or any(parent.glob("*.bin")) or (parent / "model.safetensors.index.json").is_file()
            if has_w:
                p_norm = parent.as_posix().lower().replace("-", "").replace("_", "")
                score = sum(5 for h in norm_hints if h in p_norm)
                is_input = 1 if str(parent).startswith("/kaggle/input") else 0
                candidates.append((score, is_input, -len(parent.parts), parent))

    if candidates:
        candidates.sort(reverse=True)
        chosen = candidates[0][3]
        return chosen
    
    raise FileNotFoundError(
        f"Could not auto-discover local model snapshot for {model_key} with hints {raw_hints}. "
        "Attach the Kaggle Model or set LOCAL_MODEL_PATH explicitly in parameters."
    )

def load_task_model(config: Any, logger: logging.Logger, model_key: str | None = None) -> dict:
    from transformers import AutoProcessor, AutoConfig

    # Monkeypatch Qwen3VLMoeTextConfig in transformers where pad_token_id attribute is missing
    try:
        from transformers.models.qwen3_vl_moe.configuration_qwen3_vl_moe import Qwen3VLMoeTextConfig
        if not hasattr(Qwen3VLMoeTextConfig, "pad_token_id"):
            Qwen3VLMoeTextConfig.pad_token_id = None
    except Exception:
        pass

    try:
        from transformers.models.qwen3_vl.configuration_qwen3_vl import Qwen3VLTextConfig
        if not hasattr(Qwen3VLTextConfig, "pad_token_id"):
            Qwen3VLTextConfig.pad_token_id = None
    except Exception:
        pass

    model_key = model_key or str(cfg_value(config, "ACTIVE_MODEL_KEY", "qwen3_vl_8b"))
    registry = cfg_value(config, "MODEL_REGISTRY", {})
    spec = registry[model_key]
    spec["prompt"] = cfg_value(config, "VLM_FACTORS_PROMPT")

    model_path = resolve_local_model_path(config, model_key)
    logger.info("Loading model key=%s directly from read-only mount: %s (NO copy to /kaggle/working)", model_key, model_path)

    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
    device_cfg = cfg_value(config, "DEVICE", "auto")
    device_map = "auto" if device_cfg in ("auto", "cuda") else device_cfg

    # Load and patch AutoConfig
    model_config = AutoConfig.from_pretrained(str(model_path), local_files_only=True)
    if hasattr(model_config, "text_config") and model_config.text_config is not None:
        pad_id = getattr(model_config, "pad_token_id", None) or getattr(model_config.text_config, "eos_token_id", None) or 151643
        setattr(model_config.text_config, "pad_token_id", pad_id)
    if getattr(model_config, "pad_token_id", None) is None:
        model_config.pad_token_id = getattr(getattr(model_config, "text_config", None), "pad_token_id", 151643)

    backend = spec.get("backend", "qwen3_vl")
    model_cls = None

    mtype = getattr(model_config, "model_type", "").lower()
    is_moe = (mtype == "qwen3_vl_moe") or ("moe" in model_key.lower()) or ("a3b" in model_key.lower())

    if is_moe:
        try:
            from transformers import Qwen3VLMoeForConditionalGeneration
            model_cls = Qwen3VLMoeForConditionalGeneration
        except Exception:
            pass

    if model_cls is None and (mtype == "qwen3_vl" or backend == "qwen3_vl"):
        try:
            from transformers import Qwen3VLForConditionalGeneration
            model_cls = Qwen3VLForConditionalGeneration
        except Exception:
            pass

    if model_cls is None and (mtype == "qwen2_5_vl" or backend == "qwen25_vl"):
        try:
            from transformers import Qwen2_5_VLForConditionalGeneration
            model_cls = Qwen2_5_VLForConditionalGeneration
        except Exception:
            pass

    if model_cls is None:
        try:
            from transformers import AutoModelForImageTextToText
            model_cls = AutoModelForImageTextToText
        except Exception:
            try:
                from transformers import AutoModelForCausalLM
                model_cls = AutoModelForCausalLM
            except Exception:
                from transformers import AutoModel
                model_cls = AutoModel

    logger.info("Instantiating model class: %s", model_cls.__name__)

    # Patch MoE expert weights orientation between Qwen checkpoint (in_features, out_features)
    # and transformers standard nn.Parameter (out_features, in_features).
    def _patch_moe_loading():
        def _should_transpose(key_or_name: str, sh: Any) -> bool:
            if not (sh and len(sh) == 3):
                return False
            if "mlp.experts.down_proj" in key_or_name:
                return (sh[-1] == 2048 and sh[-2] != 2048) or (sh[-1] > sh[-2])
            if "mlp.experts.gate_up_proj" in key_or_name:
                return (sh[-1] == 1536 and sh[-2] != 1536) or (sh[-2] > sh[-1])
            return False

        try:
            import accelerate.utils.modeling as acc_mod
            if not getattr(acc_mod, "_moe_transpose_patched", False):
                _orig_set = acc_mod.set_module_tensor_to_device
                def _patched_set(module, tensor_name, device, value=None, dtype=None, **kwargs):
                    if value is not None and hasattr(value, "ndim") and _should_transpose(tensor_name, value.shape):
                        value = value.transpose(-1, -2).contiguous()
                    return _orig_set(module, tensor_name, device, value=value, dtype=dtype, **kwargs)
                acc_mod.set_module_tensor_to_device = _patched_set
                acc_mod._moe_transpose_patched = True
        except Exception:
            pass

        try:
            import safetensors.torch as st_torch
            if not getattr(st_torch, "_moe_transpose_patched", False):
                _orig_st_load = st_torch.load_file
                def _patched_st_load(filename, device="cpu"):
                    sd = _orig_st_load(filename, device=device)
                    for k in list(sd.keys()):
                        v = sd[k]
                        if hasattr(v, "ndim") and _should_transpose(k, v.shape):
                            sd[k] = v.transpose(-1, -2).contiguous()
                    return sd
                st_torch.load_file = _patched_st_load
                st_torch._moe_transpose_patched = True
        except Exception:
            pass

        try:
            import safetensors
            if not getattr(safetensors, "_moe_transpose_patched", False):
                _orig_safe_open = safetensors.safe_open
                class _PatchedSlice:
                    def __init__(self, key, s_obj):
                        self._key = key
                        self._s = s_obj
                    def get_shape(self):
                        sh = list(self._s.get_shape())
                        if _should_transpose(self._key, sh):
                            sh[-2], sh[-1] = sh[-1], sh[-2]
                        return sh
                    def get_dtype(self):
                        return self._s.get_dtype()
                    def __getitem__(self, item):
                        val = self._s[item]
                        if hasattr(val, "ndim") and _should_transpose(self._key, val.shape):
                            return val.transpose(-1, -2).contiguous()
                        return val
                    def __getattr__(self, name):
                        return getattr(self._s, name)

                class _PatchedSafeOpen:
                    def __init__(self, *args, **kwargs):
                        self._f = _orig_safe_open(*args, **kwargs)
                    def __enter__(self):
                        self._ctx = self._f.__enter__()
                        return self
                    def __exit__(self, *args):
                        return self._ctx.__exit__(*args)
                    def get_tensor(self, key):
                        val = self._ctx.get_tensor(key)
                        if hasattr(val, "ndim") and _should_transpose(key, val.shape):
                            return val.transpose(-1, -2).contiguous()
                        return val
                    def get_slice(self, key):
                        return _PatchedSlice(key, self._ctx.get_slice(key))
                    def keys(self):
                        return self._ctx.keys()
                    def metadata(self):
                        return self._ctx.metadata()
                    def __getattr__(self, name):
                        return getattr(self._ctx, name)

                safetensors.safe_open = _PatchedSafeOpen
                safetensors._moe_transpose_patched = True
        except Exception:
            pass

        try:
            import transformers.modeling_utils as hf_mod
            if not getattr(hf_mod, "_moe_transpose_patched", False):
                _orig_load_state_dict = hf_mod.load_state_dict
                def _patched_load_state_dict(checkpoint_file, *args, **kwargs):
                    sd = _orig_load_state_dict(checkpoint_file, *args, **kwargs)
                    for k in list(sd.keys()):
                        v = sd[k]
                        if hasattr(v, "ndim") and _should_transpose(k, v.shape):
                            sd[k] = v.transpose(-1, -2).contiguous()
                    return sd
                hf_mod.load_state_dict = _patched_load_state_dict
                hf_mod._moe_transpose_patched = True
        except Exception:
            pass

    _patch_moe_loading()

    model = model_cls.from_pretrained(
        str(model_path),
        config=model_config,
        torch_dtype=dtype,
        device_map=device_map,
        local_files_only=True,
        ignore_mismatched_sizes=True,
    )
    model.eval()

    # Verify MoE expert layer weights if model has experts
    try:
        sample_layer = getattr(model, "language_model", model).layers[0].mlp.experts
        down_w = sample_layer.down_proj
        gate_w = sample_layer.gate_up_proj
        logger.info(
            "MoE expert layer 0 weights verified: down_proj=%s (norm=%.2f), gate_up_proj=%s (norm=%.2f)",
            tuple(down_w.shape), float(down_w.norm().item()),
            tuple(gate_w.shape), float(gate_w.norm().item()),
        )
    except Exception:
        pass

    processor = None
    try:
        processor = AutoProcessor.from_pretrained(str(model_path), local_files_only=True)
    except Exception as e:
        logger.warning("AutoProcessor.from_pretrained failed (%s), attempting explicit Qwen processor: %s", type(e).__name__, e)
        try:
            from transformers import Qwen2_5_VLProcessor
            processor = Qwen2_5_VLProcessor.from_pretrained(str(model_path), local_files_only=True)
        except Exception:
            pass

    if processor is None:
        raise RuntimeError(f"Failed to instantiate processor for {model_key} from {model_path}")

    input_device = _first_model_device(model)
    logger.info("Successfully loaded model (%s) on device=%s, dtype=%s", model_cls.__name__, input_device, dtype)
    return {
        "model_key": model_key,
        "model": model,
        "processor": processor,
        "spec": spec,
        "dtype": dtype,
        "device": input_device,
    }

def extract_batch_factors(batch_records: list[dict], ctx: dict) -> list[dict]:
    import qwen_vl_utils

    model = ctx["model"]
    processor = ctx["processor"]
    spec = ctx["spec"]
    dtype = ctx["dtype"]
    device = ctx["device"]

    valid_items = []
    for r in batch_records:
        lp = r.get("local_image_path")
        if lp and Path(lp).is_file():
            valid_items.append(r)

    if not valid_items:
        return []

    messages_batch = [_build_qwen_messages(r["local_image_path"], spec) for r in valid_items]

    texts = [
        processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
        for msg in messages_batch
    ]

    image_inputs, video_inputs = qwen_vl_utils.process_vision_info(messages_batch)

    inputs = processor(
        text=texts,
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    inputs = _move_batch_to_device(inputs, device=device, dtype=dtype)

    max_new_tokens = int(spec.get("max_new_tokens", 512))
    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    # Trim input prompt tokens from generation output
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs["input_ids"], generated_ids)
    ]
    output_texts = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )

    results = []
    for r, raw_text in zip(valid_items, output_texts):
        factors = safe_parse_json(raw_text)
        results.append({
            "record": r,
            "raw_text": raw_text,
            "factors": factors,
        })
    return results

def run_benchmark(config: Any) -> tuple[pd.DataFrame, pd.DataFrame]:
    logger = init_logger("BenchmarkLogger")
    model_keys = cfg_value(config, "BENCHMARK_MODEL_KEYS", ["qwen3_vl_8b"])
    max_frames = int(cfg_value(config, "BENCHMARK_MAX_FRAMES", 10))

    batches = [str(batch).upper() for batch in cfg_value(config, "BATCHES", [])]
    root, records = discover_offline_frame_records(config, batches, max_frames=max_frames)
    valid_records = [r for r in records if r.get("local_image_path") and Path(r["local_image_path"]).is_file()][:max_frames]

    if not valid_records:
        logger.warning("No valid local images found to run benchmark.")
        return pd.DataFrame(), pd.DataFrame()

    samples_rows = []
    summary_rows = []

    for key in model_keys:
        logger.info("Running offline benchmark for model %s...", key)
        ctx = load_task_model(config, logger, model_key=key)
        batch_size = int(ctx["spec"].get("batch_size", 16))

        # Warmup with single frame
        logger.info("Running warmup inference...")
        extract_batch_factors(valid_records[:1], ctx)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        logger.info("Benchmarking on %d frames (batch_size=%d)...", len(valid_records), batch_size)
        start_t = time.perf_counter()
        count = 0

        for b in iter_batches(valid_records, batch_size):
            out = extract_batch_factors(b, ctx)
            for item in out:
                rec = item["record"]
                factors = item["factors"]
                samples_rows.append({
                    "model_key": key,
                    "keyframe_id": rec["keyframe_id"],
                    "subjects_count": len(factors.get("subjects", [])),
                    "objects_count": len(factors.get("objects", [])),
                    "actions_count": len(factors.get("actions", [])),
                    "text_cues_count": len(factors.get("text_cues", [])),
                    "parse_error": factors.get("_parse_error", False),
                    "raw_preview": item["raw_text"][:120].replace("\n", " "),
                })
            count += len(out)

        if torch.cuda.is_available():
            torch.cuda.synchronize()
        total_sec = time.perf_counter() - start_t
        fps = count / total_sec if total_sec > 0 else 0
        sec_per_frame = total_sec / count if count > 0 else 0

        summary_rows.append({
            "model_key": key,
            "model_name": ctx["spec"]["name"],
            "frames_tested": count,
            "total_seconds": round(total_sec, 2),
            "fps": round(fps, 2),
            "seconds_per_frame": round(sec_per_frame, 3),
            "batch_size": batch_size,
        })

        # Free GPU memory before next candidate
        del ctx
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return pd.DataFrame(samples_rows), pd.DataFrame(summary_rows)

def run_pipeline(config: Any, run_kind: str = "full") -> dict:
    layout = make_run_layout(config, run_kind)
    logger = setup_logging(layout)
    logger.info("Starting offline extraction pipeline: kind=%s, run_id=%s", run_kind, layout.run_id)

    if run_kind == "full":
        confirm = str(cfg_value(config, "CONFIRM_FULL_RUN", "")).strip()
        if confirm != "RUN_FULL_DATASET":
            raise RuntimeError(
                f"Full run aborted: CONFIRM_FULL_RUN must be set to 'RUN_FULL_DATASET'. Current: '{confirm}'"
            )

    batches = [str(batch).upper() for batch in cfg_value(config, "BATCHES", [])]
    max_frames = cfg_value(config, "FULL_MAX_FRAMES" if run_kind == "full" else "DEMO_MAX_FRAMES")
    dataset_root, records = discover_offline_frame_records(config, batches, max_frames=max_frames)

    logger.info("Discovered %d frame records across batches %s from %s", len(records), batches, dataset_root)
    valid_records = [r for r in records if r.get("local_image_path") and Path(r["local_image_path"]).is_file()]
    logger.info("Valid image paths found: %d / %d", len(valid_records), len(records))

    if not valid_records:
        raise FileNotFoundError("No valid image files found to process. Please check dataset mirror mount.")

    processed_ids = set()
    if cfg_value(config, "SKIP_EXISTING", True):
        processed_ids = load_processed_keyframes_local(config)
        logger.info("Found %d already processed keyframes. Skipping them.", len(processed_ids))

    to_process = [r for r in valid_records if r["keyframe_id"] not in processed_ids]
    logger.info("Remaining frames to process: %d", len(to_process))

    if not to_process:
        logger.info("All frames have already been processed.")
        return {"status": "ALREADY_COMPLETED", "total_frames": len(valid_records)}

    ctx = load_task_model(config, logger)
    batch_size = cfg_value(config, "PIPELINE_BATCH_SIZE") or ctx["spec"].get("batch_size", 16)
    batch_size = int(batch_size)

    total_extracted = 0
    total_errors = 0
    start_time = time.perf_counter()

    iterator = iter_batches(to_process, batch_size)
    if cfg_value(config, "USE_TQDM", True):
        from tqdm.auto import tqdm
        iterator = tqdm(iterator, total=(len(to_process) + batch_size - 1) // batch_size, desc=f"{run_kind.capitalize()} Extraction")

    for batch in iterator:
        batch_start = time.perf_counter()
        try:
            results = extract_batch_factors(batch, ctx)
            annotations = []
            for item in results:
                rec = item["record"]
                factors = item["factors"]
                anno = base_annotation(rec, config, kind=cfg_value(config, "EXTRACTOR_NAME", "semantic_factors"), run_id=layout.run_id)
                anno["json_value"] = factors
                anno["ocr_texts"] = factors.get("text_cues", [])
                anno["detected_objects"] = factors.get("objects", [])
                anno["raw_response"] = item["raw_text"]
                annotations.append(anno)

            append_jsonl(layout.annotations_path, annotations)
            total_extracted += len(annotations)

            batch_dur = time.perf_counter() - batch_start
            append_metric(layout.metrics_path, {
                "timestamp": utc_now(),
                "batch_size": len(batch),
                "duration_sec": round(batch_dur, 3),
                "fps": round(len(batch) / batch_dur, 2) if batch_dur > 0 else 0,
                "total_extracted": total_extracted,
            })
        except Exception as exc:
            total_errors += len(batch)
            logger.error("Error processing batch: %s", exc)
            err_records = [{"record": r, "error": str(exc), "timestamp": utc_now()} for r in batch]
            append_jsonl(layout.errors_path, err_records)

    total_time = time.perf_counter() - start_time
    fps = total_extracted / total_time if total_time > 0 else 0

    summary = {
        "run_id": layout.run_id,
        "kind": run_kind,
        "status": "COMPLETED",
        "batches": batches,
        "total_planned": len(to_process),
        "total_extracted": total_extracted,
        "total_errors": total_errors,
        "duration_seconds": round(total_time, 2),
        "overall_fps": round(fps, 2),
        "annotations_path": str(layout.annotations_path),
        "metrics_path": str(layout.metrics_path),
        "model_key": ctx["model_key"],
        "model_name": ctx["spec"]["name"],
    }
    write_json(layout.summary_path, summary)
    logger.info("Run finished successfully: %d extracted, %d errors in %.2fs (%.2f fps)", total_extracted, total_errors, total_time, fps)
    return summary

def run_demo(config: Any) -> dict:
    return run_pipeline(config, run_kind="demo")

def run_full(config: Any) -> dict:
    return run_pipeline(config, run_kind="full")


## 5. Offline Dry Run

Kiểm tra phát hiện dataset, đọc manifest và đường dẫn ảnh local mà không nạp model.

In [5]:
# dry_summary = dry_run(cfg, max_frames=cfg.DRY_RUN_MAX_FRAMES)
# dry_summary


## 6. Offline Benchmark

Đo throughput (FPS, giây/frame) và kiểm tra chất lượng trích xuất 7 trường trên GPU RTX Pro 6000.

In [6]:
# bench_samples_df, bench_summary_df = run_benchmark(cfg)
# print("=== Benchmark Summary ===")
# display(bench_summary_df)

# if not bench_samples_df.empty:
#     print("\n=== Sample Extractions (First 5 Rows) ===")
#     display(bench_samples_df.head(5))


## 7. Offline Demo Run

Chạy thử nghiệm end-to-end trên mẫu nhỏ (`DEMO_MAX_FRAMES`) và lưu artifacts cục bộ.

In [7]:
# demo_summary = run_demo(cfg)
# demo_summary


## 8. Offline Full Run

Chạy trên toàn bộ batch sau khi đặt `CONFIRM_FULL_RUN = "RUN_FULL_DATASET"` trong parameter cell.

In [8]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [9]:
full_summaries = run_full(cfg)
full_summaries


2026-09-18 11:22:13,738 INFO Starting offline extraction pipeline: kind=full, run_id=full_20260918T112213Z_e9e39b4c


Loaded full manifest for batch=L27: run_id=full_l27_20260715T014659Z_0e797b31/shot_segments.csv (8964 frames)
Loaded full manifest for batch=L28: run_id=full_l28_20260715T021710Z_43151145/shot_segments.csv (16002 frames)


2026-09-18 11:23:13,078 INFO Discovered 24966 frame records across batches ['L27', 'L28'] from /kaggle/input/datasets/lcdngthnh/aic-2026
2026-09-18 11:23:22,254 INFO Valid image paths found: 24966 / 24966
2026-09-18 11:23:22,255 INFO Found 0 already processed keyframes. Skipping them.
2026-09-18 11:23:22,257 INFO Remaining frames to process: 24966
2026-09-18 11:23:24,125 INFO Loading model key=qwen3_vl_4b directly from read-only mount: /kaggle/input/models/qwen-lm/qwen-3-vl/transformers/8b-instruct/1 (NO copy to /kaggle/working)
2026-09-18 11:23:24,188 INFO Instantiating model class: Qwen3VLForConditionalGeneration


Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

2026-09-18 11:26:32,471 INFO Successfully loaded model (Qwen3VLForConditionalGeneration) on device=cuda:0, dtype=torch.bfloat16


Full Extraction:   0%|          | 0/261 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
2026-09-18 15:22:41,968 INFO Run finished successfully: 24966 extracted, 0 errors in 14169.50s (1.76 fps)


{'run_id': 'full_20260918T112213Z_e9e39b4c',
 'kind': 'full',
 'status': 'COMPLETED',
 'batches': ['L27', 'L28'],
 'total_planned': 24966,
 'total_extracted': 24966,
 'total_errors': 0,
 'duration_seconds': 14169.5,
 'overall_fps': 1.76,
 'annotations_path': '/kaggle/working/feature_extractor_runs/full_20260918T112213Z_e9e39b4c/artifacts/annotations.jsonl',
 'metrics_path': '/kaggle/working/feature_extractor_runs/full_20260918T112213Z_e9e39b4c/artifacts/metrics.csv',
 'model_key': 'qwen3_vl_4b',
 'model_name': 'Qwen3-VL-4B-Instruct'}

## 9. Inspect Local Artifacts

Kiểm tra kích thước file kết quả và hiển thị bản ghi JSON đầu tiên trong `annotations.jsonl`.

In [10]:
run_root = Path(cfg.RUN_ROOT)
latest = sorted([p for p in run_root.glob("*") if p.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True)[:5]
if not latest:
    print(f"No runs found under {run_root}")
for path in latest:
    print("Run directory:", path)
    for artifact in ["artifacts/summary.json", "artifacts/annotations.jsonl", "artifacts/errors.jsonl", "artifacts/metrics.csv", "run.log"]:
        cand = path / artifact
        if cand.exists():
            print(f"  {cand} ({cand.stat().st_size:,} bytes)")
    ann_file = path / "artifacts/annotations.jsonl"
    if ann_file.exists() and ann_file.stat().st_size > 0:
        print("\nSample Record from", ann_file.name, ":")
        with ann_file.open("r", encoding="utf-8") as f:
            first_line = f.readline()
            if first_line:
                print(json.dumps(json.loads(first_line), indent=2, ensure_ascii=False))


Run directory: /kaggle/working/feature_extractor_runs/full_20260918T112213Z_e9e39b4c
  /kaggle/working/feature_extractor_runs/full_20260918T112213Z_e9e39b4c/artifacts/summary.json (567 bytes)
  /kaggle/working/feature_extractor_runs/full_20260918T112213Z_e9e39b4c/artifacts/annotations.jsonl (68,262,905 bytes)
  /kaggle/working/feature_extractor_runs/full_20260918T112213Z_e9e39b4c/artifacts/metrics.csv (12,949 bytes)
  /kaggle/working/feature_extractor_runs/full_20260918T112213Z_e9e39b4c/run.log (973 bytes)

Sample Record from annotations.jsonl :
{
  "dataset_id": "ai_challenge_2025",
  "batch_id": "L27",
  "video_id": "L27_V001",
  "keyframe_id": "L27_V001_F000000",
  "frame_id": "L27_V001_F000000",
  "shot_id": "L27_V001_S0000",
  "frame_idx": 0,
  "frame_sec": 0.0,
  "timestamp_ms": 0,
  "image_gcs_uri": "gs://aic_ai_2026/processed/keyframes/dataset=ai_challenge_2025/batch=L27/profile=autoshot_v1/video_id=L27_V001/shot_0000_first_f000000.jpg",
  "kind": "semantic_factors",
  "caption